# Hotel Bar Inventory Forecasting & Par Level Recommendation System

## 1. Project Objective

This project develops a demand forecasting and inventory policy recommendation
system for hotel bars.

The system transforms transaction-level inventory records into daily demand
time series, evaluates forecasting models, calculates safety stock and reorder
points, and simulates inventory policies against historical demand.

## 2. Environment and Imports

In [ ]:
import os
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 3. Load Raw Data

In [ ]:
if "PROJECT_ROOT" not in globals():
    from pathlib import Path

    def _find_project_root(start: Path) -> Path:
        for parent in [start] + list(start.parents):
            if (parent / "data" / "raw" / "bar_inventory_data.csv").exists():
                return parent
        raise FileNotFoundError("PROJECT_ROOT not found")

    PROJECT_ROOT = _find_project_root(Path.cwd())

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "bar_inventory_data.csv"

df = pd.read_csv(RAW_DATA_PATH)

print(df.shape)
print(df.columns.tolist())

display(df.head())


In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

## 4. Data Audit & Validation

Before transforming the transaction-level records into a daily demand
time series, we validate timestamps, categorical fields, numeric values,
duplicates, and the inventory conservation equation.

In [ ]:
df["Date Time Served"] = pd.to_datetime(
    df["Date Time Served"],
    errors="coerce"
)

In [ ]:
print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")

print(f"\nDate range:")
print(f"Start: {df['Date Time Served'].min()}")
print(f"End:   {df['Date Time Served'].max()}")

print(f"\nUnique bars: {df['Bar Name'].nunique()}")
print(f"Unique alcohol types: {df['Alcohol Type'].nunique()}")
print(f"Unique brands: {df['Brand Name'].nunique()}")

In [ ]:
print("Bars:")
display(df["Bar Name"].value_counts())

print("\nAlcohol Types:")
display(df["Alcohol Type"].value_counts())

print("\nBrands:")
display(df["Brand Name"].value_counts())

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)

display(missing)

print("Total missing values:", df.isna().sum().sum())

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_count:,}")
print(
    f"Duplicate percentage: "
    f"{duplicate_count / len(df) * 100:.2f}%"
)

In [ ]:
duplicates = df[df.duplicated(keep=False)].sort_values(
    by=["Date Time Served", "Bar Name", "Brand Name"]
)

display(duplicates.head(20))

In [ ]:
numeric_cols = [
    "Opening Balance (ml)",
    "Purchase (ml)",
    "Consumed (ml)",
    "Closing Balance (ml)"
]

display(df[numeric_cols].describe().T)

negative_counts = (df[numeric_cols] < 0).sum()

display(negative_counts)

In [ ]:
df["Expected Closing (ml)"] = (
    df["Opening Balance (ml)"]
    + df["Purchase (ml)"]
    - df["Consumed (ml)"]
)

df["Balance Difference (ml)"] = (
    df["Closing Balance (ml)"]
    - df["Expected Closing (ml)"]
)

display(
    df["Balance Difference (ml)"]
    .describe()
)

print(
    "Rows satisfying conservation exactly:",
    (df["Balance Difference (ml)"] == 0).sum()
)

print(
    "Rows violating conservation:",
    (df["Balance Difference (ml)"] != 0).sum()
)

In [ ]:
TOLERANCE_ML = 0.01

df["Conservation Valid"] = (
    df["Balance Difference (ml)"].abs()
    <= TOLERANCE_ML
)

validation_summary = {
    "total_rows": len(df),
    "valid_rows": df["Conservation Valid"].sum(),
    "invalid_rows": (~df["Conservation Valid"]).sum(),
    "valid_percentage": df["Conservation Valid"].mean() * 100,
}

validation_summary

In [ ]:
invalid_balance = df[
    ~df["Conservation Valid"]
].copy()

display(
    invalid_balance[
        [
            "Date Time Served",
            "Bar Name",
            "Alcohol Type",
            "Brand Name",
            "Opening Balance (ml)",
            "Purchase (ml)",
            "Consumed (ml)",
            "Closing Balance (ml)",
            "Expected Closing (ml)",
            "Balance Difference (ml)",
        ]
    ].head(20)
)

In [ ]:
brand_type_counts = (
    df.groupby("Brand Name")["Alcohol Type"]
    .nunique()
    .sort_values(ascending=False)
)

display(brand_type_counts)

multi_type_brands = brand_type_counts[
    brand_type_counts > 1
]

display(multi_type_brands)

In [ ]:
sample_bar = df["Bar Name"].iloc[0]
sample_brand = df["Brand Name"].iloc[0]

print("Sample Bar x Brand:", sample_bar, "x", sample_brand)

sample = (
    df[
        (df["Bar Name"] == sample_bar) &
        (df["Brand Name"] == sample_brand)
    ]
    .sort_values("Date Time Served")
)

display(sample.head(20))

In [ ]:
bar_brand = (
    df.groupby(
        ["Bar Name", "Brand Name"]
    )
    .agg(
        transactions=("Brand Name", "size"),
        first_date=("Date Time Served", "min"),
        last_date=("Date Time Served", "max"),
        total_consumption_ml=("Consumed (ml)", "sum"),
        total_purchase_ml=("Purchase (ml)", "sum"),
    )
    .reset_index()
)

display(bar_brand.head())

print(
    "Number of Bar x Brand combinations:",
    len(bar_brand)
)

In [ ]:
print(
    "Total consumption:",
    df["Consumed (ml)"].sum()
)

display(
    df["Consumed (ml)"].describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

print(
    "Zero-consumption transactions:",
    (df["Consumed (ml)"] == 0).sum()
)

print(
    "Zero-consumption percentage:",
    (df["Consumed (ml)"] == 0).mean() * 100
)

In [ ]:
daily_transactions = (
    df.assign(
        Date=df["Date Time Served"].dt.date
    )
    .groupby("Date")
    .size()
)

display(daily_transactions.describe())

plt.figure(figsize=(14, 5))

daily_transactions.plot()

plt.title("Number of Transactions per Day")
plt.xlabel("Date")
plt.ylabel("Transactions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Data Processing: Build Daily Demand Dataset

Pipeline: Raw CSV -> Validated records -> Daily aggregation -> Missing dates
filled -> Daily demand dataset -> EDA.

Important: a missing date does not automatically mean zero consumption. We
first measure date coverage per Bar x Brand, then decide how to complete the
calendar. Inventory balances are never blindly filled with zero.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_processing import (
    prepare_raw_data,
    aggregate_daily_consumption
)

processed_df = prepare_raw_data(df)

daily_df = aggregate_daily_consumption(processed_df)

display(daily_df.head())

In [ ]:
coverage = (
    daily_df
    .groupby(["Bar Name", "Brand Name"])
    .agg(
        first_date=("Date", "min"),
        last_date=("Date", "max"),
        observed_days=("Date", "nunique")
    )
    .reset_index()
)

coverage["calendar_days"] = (
    coverage["last_date"] - coverage["first_date"]
).dt.days + 1

coverage["coverage_pct"] = (
    coverage["observed_days"]
    / coverage["calendar_days"]
    * 100
)

display(
    coverage.sort_values("coverage_pct").head(20)
)

In [ ]:
def create_daily_calendar(daily_df: pd.DataFrame) -> pd.DataFrame:

    series_list = []

    for (bar, brand), group in daily_df.groupby(
        ["Bar Name", "Brand Name"]
    ):

        group = group.sort_values("Date").copy()

        full_dates = pd.date_range(
            start=group["Date"].min(),
            end=group["Date"].max(),
            freq="D"
        )

        group = (
            group
            .set_index("Date")
            .reindex(full_dates)
            .rename_axis("Date")
            .reset_index()
        )

        group["Bar Name"] = bar
        group["Brand Name"] = brand

        series_list.append(group)

    result = pd.concat(
        series_list,
        ignore_index=True
    )

    return result

In [ ]:
daily_complete = create_daily_calendar(daily_df)

display(daily_complete.head(20))

In [ ]:
daily_complete["daily_consumption_ml"] = (
    daily_complete["daily_consumption_ml"]
    .fillna(0)
)

daily_complete["daily_purchase_ml"] = (
    daily_complete["daily_purchase_ml"]
    .fillna(0)
)

# NOTE: opening/closing inventory are intentionally left as NaN on
# completed days -- missing inventory is NOT zero inventory.

In [ ]:
daily_complete["day_of_week"] = (
    daily_complete["Date"].dt.dayofweek
)

daily_complete["day_name"] = (
    daily_complete["Date"].dt.day_name()
)

daily_complete["is_weekend"] = (
    daily_complete["day_of_week"] >= 5
)

daily_complete["week_of_year"] = (
    daily_complete["Date"].dt.isocalendar().week.astype(int)
)

daily_complete["month"] = (
    daily_complete["Date"].dt.month
)

In [ ]:
daily_consumption = daily_complete[
    [
        "Date",
        "Bar Name",
        "Brand Name",
        "daily_consumption_ml",
        "daily_purchase_ml",
        "opening_inventory_ml",
        "closing_inventory_ml",
        "record_count",
        "day_of_week",
        "day_name",
        "is_weekend",
        "week_of_year",
        "month"
    ]
].copy()

daily_consumption = daily_consumption.sort_values(
    ["Bar Name", "Brand Name", "Date"]
).reset_index(drop=True)

In [ ]:
daily_consumption.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "daily_bar_consumption.csv"),
    index=False
)

print(daily_consumption.shape)
display(daily_consumption.head())

In [ ]:
print(
    "Negative consumption:",
    (daily_consumption["daily_consumption_ml"] < 0).sum()
)

raw_total = df["Consumed (ml)"].sum()

daily_total = (
    daily_consumption["daily_consumption_ml"].sum()
)

print("Raw total:", raw_total)
print("Daily total:", daily_total)
print("Difference:", raw_total - daily_total)

print(
    daily_consumption["Date"].min(),
    "to",
    daily_consumption["Date"].max()
)

print(
    daily_consumption
    .groupby(["Bar Name", "Brand Name"])
    .size()
    .describe()
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

if "PROJECT_ROOT" not in globals():
    from pathlib import Path

    def _find_project_root(start: Path) -> Path:
        for p in [start] + list(start.parents):
            if (p / "data" / "raw" / "bar_inventory_data.csv").exists():
                return p
        return start

    PROJECT_ROOT = _find_project_root(Path.cwd())

daily = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "daily_bar_consumption.csv",
    parse_dates=["Date"]
)

print(daily.shape)
display(daily.head())

In [ ]:
print("Date range:")
print(daily["Date"].min(), "to", daily["Date"].max())

print("\nBars:", daily["Bar Name"].nunique())
print("Brands:", daily["Brand Name"].nunique())

print(
    "Bar × Brand combinations:",
    daily[["Bar Name", "Brand Name"]].drop_duplicates().shape[0]
)

## 6. Demand Analysis & EDA

Pipeline: daily demand dataset -> data quality -> overall consumption (total, by bar, by brand, by bar x brand) -> demand patterns (daily, weekly, weekend, monthly, variability) -> series classification (velocity / intermittent) -> forecasting strategy.

In [ ]:
if "PROJECT_ROOT" not in globals():
    from pathlib import Path

    def _find_project_root(start: Path) -> Path:
        for p in [start] + list(start.parents):
            if (p / "data" / "raw" / "bar_inventory_data.csv").exists():
                return p
        return start

    PROJECT_ROOT = _find_project_root(Path.cwd())

daily = pd.read_csv(
    str(PROJECT_ROOT / "data" / "processed" / "daily_bar_consumption.csv"),
    parse_dates=["Date"]
)

print(daily.shape)
display(daily.head())

In [ ]:
print("Date range:")
print(daily["Date"].min(), "to", daily["Date"].max())

print("\nBars:", daily["Bar Name"].nunique())
print("Brands:", daily["Brand Name"].nunique())

print(
    "Bar x Brand combinations:",
    daily[["Bar Name", "Brand Name"]].drop_duplicates().shape[0]
)

In [ ]:
daily.info()

In [ ]:
display(daily.describe(include="all"))

In [ ]:
total_consumption = daily["daily_consumption_ml"].sum()

print(
    f"Total consumption: {total_consumption:,.2f} ml"
)

average_daily_consumption = (
    daily.groupby("Date")["daily_consumption_ml"]
    .sum()
    .mean()
)

print(
    f"Average daily consumption: "
    f"{average_daily_consumption:,.2f} ml"
)

In [ ]:
daily_total = (
    daily.groupby("Date")["daily_consumption_ml"]
    .sum()
    .reset_index()
)

display(daily_total.head())

plt.figure(figsize=(14, 5))

plt.plot(
    daily_total["Date"],
    daily_total["daily_consumption_ml"]
)

plt.title("Total Daily Consumption")
plt.xlabel("Date")
plt.ylabel("Consumption (ml)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
bar_consumption = (
    daily.groupby("Bar Name")["daily_consumption_ml"]
    .sum()
    .sort_values(ascending=False)
)

display(bar_consumption)

plt.figure(figsize=(10, 5))

bar_consumption.plot(kind="bar")

plt.title("Total Consumption by Bar")
plt.xlabel("Bar")
plt.ylabel("Consumption (ml)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
brand_consumption = (
    daily.groupby("Brand Name")["daily_consumption_ml"]
    .sum()
    .sort_values(ascending=False)
)

display(brand_consumption)

top_n = 15

plt.figure(figsize=(12, 6))

brand_consumption.head(top_n).sort_values().plot(
    kind="barh"
)

plt.title(f"Top {top_n} Brands by Consumption")
plt.xlabel("Consumption (ml)")

plt.tight_layout()
plt.show()

In [ ]:
abc = (
    daily.groupby("Brand Name")["daily_consumption_ml"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

abc["consumption_share"] = (
    abc["daily_consumption_ml"]
    / abc["daily_consumption_ml"].sum()
)

abc["cumulative_share"] = (
    abc["consumption_share"].cumsum()
)

def classify_abc(cumulative_share):
    if cumulative_share <= 0.80:
        return "A"
    elif cumulative_share <= 0.95:
        return "B"
    else:
        return "C"

abc["ABC_class"] = (
    abc["cumulative_share"]
    .apply(classify_abc)
)

display(abc)

In [ ]:
weekday_demand = (
    daily.groupby("day_name")["daily_consumption_ml"]
    .mean()
)

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_demand = weekday_demand.reindex(weekday_order)

display(weekday_demand)

plt.figure(figsize=(10, 5))

weekday_demand.plot(kind="bar")

plt.title("Average Consumption by Day of Week")
plt.xlabel("Day")
plt.ylabel("Average Consumption (ml)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
weekend_analysis = (
    daily.groupby("is_weekend")["daily_consumption_ml"]
    .agg(["mean", "median", "std", "sum"])
)

display(weekend_analysis)

weekday_mean = daily.loc[
    ~daily["is_weekend"],
    "daily_consumption_ml"
].mean()

weekend_mean = daily.loc[
    daily["is_weekend"],
    "daily_consumption_ml"
].mean()

print("Weekday average:", weekday_mean)
print("Weekend average:", weekend_mean)

print(
    "Weekend / Weekday ratio:",
    weekend_mean / weekday_mean
)

In [ ]:
monthly_demand = (
    daily.groupby("month")["daily_consumption_ml"]
    .mean()
)

display(monthly_demand)

plt.figure(figsize=(10, 5))

monthly_demand.plot(kind="bar")

plt.title("Average Consumption by Month")
plt.xlabel("Month")
plt.ylabel("Average Daily Consumption (ml)")

plt.tight_layout()
plt.show()

In [ ]:
series_summary = (
    daily
    .groupby(["Bar Name", "Brand Name"])
    .agg(
        total_consumption_ml=(
            "daily_consumption_ml",
            "sum"
        ),
        average_daily_consumption_ml=(
            "daily_consumption_ml",
            "mean"
        ),
        median_daily_consumption_ml=(
            "daily_consumption_ml",
            "median"
        ),
        std_daily_consumption_ml=(
            "daily_consumption_ml",
            "std"
        ),
        active_days=(
            "daily_consumption_ml",
            lambda x: (x > 0).sum()
        ),
        total_days=(
            "daily_consumption_ml",
            "size"
        )
    )
    .reset_index()
)

series_summary["active_day_pct"] = (
    series_summary["active_days"]
    / series_summary["total_days"]
    * 100
)

series_summary["coefficient_of_variation"] = (
    series_summary["std_daily_consumption_ml"]
    / series_summary["average_daily_consumption_ml"]
)

display(
    series_summary.sort_values(
        "total_consumption_ml",
        ascending=False
    ).head(20)
)

In [ ]:
series_summary["zero_day_pct"] = (
    1
    - series_summary["active_days"]
    / series_summary["total_days"]
) * 100

display(
    series_summary.sort_values(
        "zero_day_pct",
        ascending=False
    ).head(20)
)

In [ ]:
plt.figure(figsize=(14, 5))

for kind in ["High Velocity", "Medium Velocity"]:
    pass

sample_bar = daily["Bar Name"].iloc[0]
sample_brand = daily["Brand Name"].iloc[0]

sample_series = daily[
    (daily["Bar Name"] == sample_bar) &
    (daily["Brand Name"] == sample_brand)
].sort_values("Date")

plt.plot(
    sample_series["Date"],
    sample_series["daily_consumption_ml"]
)

plt.title(
    f"{sample_brand} - {sample_bar}"
)

plt.xlabel("Date")
plt.ylabel("Consumption (ml)")

plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
variability = (
    daily.groupby(["Bar Name", "Brand Name"])
    ["daily_consumption_ml"]
    .agg(
        mean="mean",
        std="std",
        min="min",
        max="max"
    )
    .reset_index()
)

variability["cv"] = (
    variability["std"]
    / variability["mean"]
)

display(
    variability.sort_values(
        "cv",
        ascending=False
    ).head(20)
)

In [ ]:
def classify_demand(row):
    if row["active_day_pct"] < 30:
        return "Intermittent"
    elif row["average_daily_consumption_ml"] < 100:
        return "Low Velocity"
    elif row["average_daily_consumption_ml"] < 500:
        return "Medium Velocity"
    else:
        return "High Velocity"

series_summary["demand_class"] = (
    series_summary.apply(
        classify_demand,
        axis=1
    )
)

print(
    series_summary["demand_class"]
    .value_counts()
)

In [ ]:
series_summary.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "series_summary.csv"),
    index=False
)

abc.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "brand_abc_analysis.csv"),
    index=False
)

print("saved series_summary.csv and brand_abc_analysis.csv")

## 6.5 Phase 3.5 — Intermittent Demand Analysis

We must not choose a forecasting model based only on the `active_day_pct < 30` rule. That rule only measures **how often** demand occurs. We also need to measure **how variable the amount is when demand actually happens**. These are two separate forecasting problems:

1. Will demand happen? (occurrence / intermittency)
2. How much will be consumed if it happens? (demand size)

This section inspects both dimensions and produces a 2-D classification (frequency / variability).


In [ ]:
display(
    series_summary[
        [
            "Bar Name",
            "Brand Name",
            "active_days",
            "total_days",
            "active_day_pct",
            "zero_day_pct"
        ]
    ]
    .sort_values("active_day_pct")
    .head(30)
)

print("\nActive-day percentage statistics:")
display(series_summary["active_day_pct"].describe())


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(
    series_summary["zero_day_pct"],
    bins=15
)

plt.xlabel("Zero-demand days (%)")
plt.ylabel("Number of Bar × Brand series")
plt.title("Distribution of Zero-Demand Days")

plt.show()


In [ ]:
top_series = (
    series_summary
    .sort_values("total_consumption_ml", ascending=False)
    .iloc[0]
)

sample_bar = top_series["Bar Name"]
sample_brand = top_series["Brand Name"]

print("Sample Bar:", sample_bar)
print("Sample Brand:", sample_brand)

sample = (
    daily[
        (daily["Bar Name"] == sample_bar) &
        (daily["Brand Name"] == sample_brand)
    ]
    .sort_values("Date")
)

display(
    sample[
        [
            "Date",
            "daily_consumption_ml",
            "daily_purchase_ml",
            "day_name"
        ]
    ].head(50)
)


In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    sample["Date"],
    sample["daily_consumption_ml"]
)

plt.xlabel("Date")
plt.ylabel("Consumption (ml)")
plt.title(f"Daily Consumption — {sample_bar} / {sample_brand}")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
active_demand = daily[
    daily["daily_consumption_ml"] > 0
].copy()

active_demand_summary = (
    active_demand
    .groupby(["Bar Name", "Brand Name"])["daily_consumption_ml"]
    .agg(
        active_mean="mean",
        active_median="median",
        active_std="std",
        active_min="min",
        active_max="max"
    )
    .reset_index()
)

active_demand_summary["active_cv"] = (
    active_demand_summary["active_std"] /
    active_demand_summary["active_mean"]
)

display(
    active_demand_summary
    .sort_values("active_cv", ascending=False)
    .head(20)
)


In [ ]:
# Self-heal: regenerate active-demand metrics if Step 4 output is missing
required_active_cols = {
    "active_mean", "active_median", "active_std",
    "active_min", "active_max", "active_cv"
}
if ("active_demand_summary" not in globals()
        or not required_active_cols.issubset(active_demand_summary.columns)):
    active_demand = daily[daily["daily_consumption_ml"] > 0].copy()
    active_demand_summary = (
        active_demand
        .groupby(["Bar Name", "Brand Name"])["daily_consumption_ml"]
        .agg(
            active_mean="mean",
            active_median="median",
            active_std="std",
            active_min="min",
            active_max="max"
        )
        .reset_index()
    )
    active_demand_summary["active_cv"] = (
        active_demand_summary["active_std"]
        / active_demand_summary["active_mean"]
    )

# Idempotency: if a previous run already merged these columns, drop them first
prior_cols = [c for c in ["active_mean", "active_median", "active_std",
                          "active_min", "active_max", "active_cv",
                          "demand_frequency", "demand_variability", "demand_class"]
              if c in series_summary.columns]
if prior_cols:
    series_summary = series_summary.drop(columns=prior_cols)

series_summary = series_summary.merge(
    active_demand_summary[
        [
            "Bar Name",
            "Brand Name",
            "active_mean",
            "active_median",
            "active_std",
            "active_min",
            "active_max",
            "active_cv"
        ]
    ],
    on=["Bar Name", "Brand Name"],
    how="left"
)

display(
    series_summary[
        [
            "Bar Name",
            "Brand Name",
            "active_day_pct",
            "zero_day_pct",
            "active_mean",
            "active_median",
            "active_cv"
        ]
    ].head(20)
)

def classify_frequency(pct):
    if pct >= 30:
        return "Frequent"
    elif pct >= 10:
        return "Occasional"
    else:
        return "Intermittent"

def classify_variability(cv):
    if cv < 0.5:
        return "Low Variability"
    elif cv < 1.0:
        return "Medium Variability"
    else:
        return "High Variability"

series_summary["demand_frequency"] = (
    series_summary["active_day_pct"].apply(classify_frequency)
)
series_summary["demand_variability"] = (
    series_summary["active_cv"].apply(classify_variability)
)
series_summary["demand_class"] = (
    series_summary["demand_frequency"]
    + " / "
    + series_summary["demand_variability"]
)

print(series_summary["demand_class"].value_counts())

if "PROJECT_ROOT" not in globals():
    from pathlib import Path

    def _find_project_root(start: Path) -> Path:
        for p in [start] + list(start.parents):
            if (p / "data" / "raw" / "bar_inventory_data.csv").exists():
                return p
        return start

    PROJECT_ROOT = _find_project_root(Path.cwd())

series_summary.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "series_summary.csv"),
    index=False
)

print("Updated series_summary.csv saved.")


In [ ]:
sample_weekday = (
    sample
    .groupby("day_name")["daily_consumption_ml"]
    .agg(
        mean="mean",
        median="median",
        total="sum"
    )
    .reindex([
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ])
)

display(sample_weekday)

sample_active_weekday = (
    sample[sample["daily_consumption_ml"] > 0]
    .groupby("day_name")["daily_consumption_ml"]
    .agg(
        active_mean="mean",
        active_median="median",
        active_total="sum",
        active_days="count"
    )
    .reindex([
        "Monday",
        "Tuesday",
        "Wednesday",
        "Thursday",
        "Friday",
        "Saturday",
        "Sunday"
    ])
)

display(sample_active_weekday)


In [ ]:
intermittent_analysis = series_summary[
    [
        "Bar Name",
        "Brand Name",
        "active_days",
        "total_days",
        "active_day_pct",
        "zero_day_pct",
        "total_consumption_ml",
        "average_daily_consumption_ml",
        "std_daily_consumption_ml",
        "coefficient_of_variation",
        "active_mean",
        "active_median",
        "active_std",
        "active_cv",
        "demand_frequency",
        "demand_variability",
        "demand_class"
    ]
].copy()

if "PROJECT_ROOT" not in globals():
    from pathlib import Path

    def _find_project_root(start: Path) -> Path:
        for p in [start] + list(start.parents):
            if (p / "data" / "raw" / "bar_inventory_data.csv").exists():
                return p
        return start

    PROJECT_ROOT = _find_project_root(Path.cwd())

intermittent_analysis.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "intermittent_demand_analysis.csv"),
    index=False
)

print("Saved:")
print(str(PROJECT_ROOT / "data" / "processed" / "intermittent_demand_analysis.csv"))


## 7. Phase 4 — Forecasting Strategy

Objective: historical daily consumption -> baseline -> intermittent-demand model -> time-based backtesting -> compare MAE / RMSE / WAPE -> select a model per Bar x Brand.

Phase 3.5 showed the overall CV around 3 was inflated by heavy zero days; active-day CVs are at most about 0.59. So a complicated ML model is not justified out of the gate. We start with simple baselines and let time-based backtesting (never random splits) decide whether extra complexity actually improves the forecast.


In [ ]:
import pandas as pd
import numpy as np

FORECAST_HORIZON = 30

def time_split_series(series, forecast_horizon=30):
    series = series.sort_values("Date").copy()

    if len(series) <= forecast_horizon:
        return None, None

    train = series.iloc[:-forecast_horizon].copy()
    test = series.iloc[-forecast_horizon:].copy()

    return train, test


In [ ]:
def mean_forecast(train, horizon):
    mean_demand = train["daily_consumption_ml"].mean()

    return np.repeat(
        mean_demand,
        horizon
    )


In [ ]:
def intermittent_mean_forecast(train, horizon):
    demand = train["daily_consumption_ml"]

    probability_demand = (demand > 0).mean()

    active_demand = demand[demand > 0]

    if len(active_demand) == 0:
        return np.zeros(horizon)

    active_mean = active_demand.mean()

    expected_daily_demand = (
        probability_demand * active_mean
    )

    return np.repeat(
        expected_daily_demand,
        horizon
    )


In [ ]:
def moving_average_forecast(train, horizon, window=14):
    demand = train["daily_consumption_ml"]

    window = min(window, len(demand))

    forecast_value = demand.tail(window).mean()

    return np.repeat(
        forecast_value,
        horizon
    )


In [ ]:
def mae(y_true, y_pred):
    return np.mean(
        np.abs(y_true - y_pred)
    )

def rmse(y_true, y_pred):
    return np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

def wape(y_true, y_pred):
    denominator = np.sum(np.abs(y_true))

    if denominator == 0:
        return np.nan

    return (
        np.sum(np.abs(y_true - y_pred))
        / denominator
    )


### 4.2 — Intermittent-demand models (Croston / SBA / TSB-style)

These models were designed for zero-heavy demand, which our data clearly is. WAPE is only compared among series where total actual demand > 0 (it is undefined otherwise).


In [ ]:
def croston_forecast(train, horizon, alpha=0.1):
    """
    Croston's method for intermittent demand.

    Separately estimates:
    - demand size
    - interval between non-zero demands
    """

    demand = train["daily_consumption_ml"].values

    non_zero = demand[demand > 0]

    if len(non_zero) == 0:
        return np.zeros(horizon)

    # First non-zero demand
    first_index = np.where(demand > 0)[0][0]

    z = demand[first_index]   # demand estimate
    p = first_index + 1       # interval estimate

    interval = 0

    for t in range(first_index + 1, len(demand)):

        if demand[t] > 0:

            interval += 1

            z = (
                alpha * demand[t]
                + (1 - alpha) * z
            )

            p = (
                alpha * interval
                + (1 - alpha) * p
            )

            interval = 0

        else:
            interval += 1

    forecast_value = z / p if p > 0 else 0

    return np.repeat(
        forecast_value,
        horizon
    )


In [ ]:
def sba_forecast(train, horizon, alpha=0.1):
    """
    Syntetos-Boylan Approximation (SBA)
    """

    demand = train["daily_consumption_ml"].values

    if np.sum(demand > 0) == 0:
        return np.zeros(horizon)

    first_index = np.where(demand > 0)[0][0]

    z = demand[first_index]
    p = first_index + 1

    interval = 0

    for t in range(first_index + 1, len(demand)):

        if demand[t] > 0:

            interval += 1

            z = (
                alpha * demand[t]
                + (1 - alpha) * z
            )

            p = (
                alpha * interval
                + (1 - alpha) * p
            )

            interval = 0

        else:
            interval += 1

    forecast_value = (
        (1 - alpha / 2)
        * z
        / p
    )

    return np.repeat(
        forecast_value,
        horizon
    )


In [ ]:
results = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    group = group.sort_values("Date").reset_index(drop=True)

    train, test = time_split_series(
        group,
        forecast_horizon=FORECAST_HORIZON
    )

    if train is None:
        continue

    y_true = test["daily_consumption_ml"].values

    forecasts = {
        "Mean": mean_forecast(train, FORECAST_HORIZON),

        "Intermittent Mean": intermittent_mean_forecast(
            train,
            FORECAST_HORIZON
        ),

        "MA-7": moving_average_forecast(
            train,
            FORECAST_HORIZON,
            window=7
        ),

        "MA-14": moving_average_forecast(
            train,
            FORECAST_HORIZON,
            window=14
        ),

        "MA-30": moving_average_forecast(
            train,
            FORECAST_HORIZON,
            window=30
        ),

        "Croston": croston_forecast(
            train,
            FORECAST_HORIZON
        ),

        "SBA": sba_forecast(
            train,
            FORECAST_HORIZON
        ),
    }

    for model_name, prediction in forecasts.items():

        results.append({
            "Bar Name": bar,
            "Brand Name": brand,
            "Model": model_name,
            "MAE": mae(y_true, prediction),
            "RMSE": rmse(y_true, prediction),
            "WAPE": wape(y_true, prediction),
        })

forecast_results = pd.DataFrame(results)

display(
    forecast_results.head()
)


In [ ]:
valid_results = forecast_results.dropna(
    subset=["WAPE"]
)

model_summary = (
    valid_results
    .groupby("Model")
    .agg(
        series_count=("Model", "size"),
        mean_MAE=("MAE", "mean"),
        median_MAE=("MAE", "median"),
        mean_RMSE=("RMSE", "mean"),
        median_RMSE=("RMSE", "median"),
        mean_WAPE=("WAPE", "mean"),
        median_WAPE=("WAPE", "median")
    )
    .sort_values("mean_WAPE")
)

display(model_summary)


In [ ]:
best_models = (
    valid_results
    .sort_values(
        ["Bar Name", "Brand Name", "WAPE"]
    )
    .groupby(
        ["Bar Name", "Brand Name"],
        as_index=False
    )
    .first()
)

display(best_models.head(20))

print(
    best_models["Model"]
    .value_counts()
)


In [ ]:
forecast_results.to_csv(
    PROJECT_ROOT / "data" / "processed" / "forecast_model_results.csv",
    index=False
)

best_models.to_csv(
    PROJECT_ROOT / "data" / "processed" / "best_forecasting_models.csv",
    index=False
)

print("Forecast evaluation results saved.")


## 8. Phase 5 — Rolling Backtest & Final Model Selection

A single 30-day holdout is a first experiment; a production model should earn its place over **several historical test windows**. Rolling-origin backtesting creates multiple 'past future' periods so the per-series model choice is defensible, not accidental.


In [ ]:
def rolling_time_splits(
    series,
    forecast_horizon=30,
    n_splits=4,
    min_train_size=90
):
    series = (
        series
        .sort_values("Date")
        .reset_index(drop=True)
    )

    total_length = len(series)

    splits = []

    for i in range(n_splits):

        test_end = (
            total_length
            - i * forecast_horizon
        )

        test_start = (
            test_end
            - forecast_horizon
        )

        if test_start < min_train_size:
            break

        train = series.iloc[:test_start].copy()

        test = series.iloc[
            test_start:test_end
        ].copy()

        splits.append(
            (train, test)
        )

    return list(reversed(splits))


In [ ]:
def generate_forecast(
    model_name,
    train,
    horizon
):

    if model_name == "Mean":
        return mean_forecast(
            train,
            horizon
        )

    elif model_name == "Intermittent Mean":
        return intermittent_mean_forecast(
            train,
            horizon
        )

    elif model_name == "MA-7":
        return moving_average_forecast(
            train,
            horizon,
            window=7
        )

    elif model_name == "MA-14":
        return moving_average_forecast(
            train,
            horizon,
            window=14
        )

    elif model_name == "MA-30":
        return moving_average_forecast(
            train,
            horizon,
            window=30
        )

    elif model_name == "Croston":
        return croston_forecast(
            train,
            horizon
        )

    elif model_name == "SBA":
        return sba_forecast(
            train,
            horizon
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )


In [ ]:
rolling_results = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    group = (
        group
        .sort_values("Date")
        .reset_index(drop=True)
    )

    splits = rolling_time_splits(
        group,
        forecast_horizon=30,
        n_splits=4,
        min_train_size=90
    )

    for fold, (train, test) in enumerate(
        splits,
        start=1
    ):

        y_true = (
            test["daily_consumption_ml"]
            .values
        )

        for model_name in [
            "Mean",
            "Intermittent Mean",
            "MA-7",
            "MA-14",
            "MA-30",
            "Croston",
            "SBA"
        ]:

            prediction = generate_forecast(
                model_name,
                train,
                len(test)
            )

            rolling_results.append({
                "Bar Name": bar,
                "Brand Name": brand,
                "Fold": fold,
                "Model": model_name,
                "MAE": mae(
                    y_true,
                    prediction
                ),
                "RMSE": rmse(
                    y_true,
                    prediction
                ),
                "WAPE": wape(
                    y_true,
                    prediction
                )
            })

rolling_results = pd.DataFrame(
    rolling_results
)

display(
    rolling_results.head()
)


In [ ]:
rolling_summary = (
    rolling_results
    .dropna(subset=["WAPE"])
    .groupby("Model")
    .agg(
        folds=("Model", "size"),
        mean_MAE=("MAE", "mean"),
        median_MAE=("MAE", "median"),
        mean_RMSE=("RMSE", "mean"),
        median_RMSE=("RMSE", "median"),
        mean_WAPE=("WAPE", "mean"),
        median_WAPE=("WAPE", "median")
    )
    .sort_values("mean_WAPE")
)

display(rolling_summary)


In [ ]:
series_model_scores = (
    rolling_results
    .dropna(subset=["WAPE"])
    .groupby(
        ["Bar Name", "Brand Name", "Model"]
    )
    .agg(
        mean_WAPE=("WAPE", "mean"),
        mean_MAE=("MAE", "mean"),
        mean_RMSE=("RMSE", "mean"),
        folds=("WAPE", "count")
    )
    .reset_index()
)

final_model_selection = (
    series_model_scores
    .sort_values(
        [
            "Bar Name",
            "Brand Name",
            "mean_WAPE"
        ]
    )
    .groupby(
        ["Bar Name", "Brand Name"],
        as_index=False
    )
    .first()
)

display(
    final_model_selection.head(20)
)

print(
    final_model_selection["Model"]
    .value_counts()
)


## 9. Phase 5 — Final 30-Day Demand Forecast

We now stop asking "which model performs best historically?" and answer: "using the selected model, what demand should we expect for each Bar × Brand over the next 30 days?"

Historical consumption -> rolling backtesting -> best model per Bar × Brand -> train selected model on ALL history -> 30-day forecast -> forecast uncertainty -> lead-time demand -> safety stock -> reorder point -> par level.

Two honest caveats carry into the inventory policy:

1. **WAPE is frequently above 1.0** (e.g., Anderson's Bar / Malibu, Intermittent Mean, WAPE ≈ 6.9). Point forecasts are inherently uncertain for highly intermittent series; we must not hide that. The policy absorbs this uncertainty through safety stock rather than pretending forecasts are exact.
2. **Observed zeros may be censored demand** — if opening stock runs out, *consumed = closing = 0* can mean the sale was constrained by inventory, not that demand was zero. Phase 6 starts with a stockout/censoring check before anything is plugged into the safety-stock formula.


### 5.1 — Train the selected model on all history


In [ ]:
FORECAST_HORIZON = 30

final_forecasts = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    group = (
        group
        .sort_values("Date")
        .reset_index(drop=True)
    )

    # Find selected model
    selected = final_model_selection[
        (final_model_selection["Bar Name"] == bar)
        & (final_model_selection["Brand Name"] == brand)
    ]

    if selected.empty:
        continue

    model_name = selected.iloc[0]["Model"]

    # Generate forecast using ALL available history
    prediction = generate_forecast(
        model_name,
        group,
        FORECAST_HORIZON
    )

    last_date = group["Date"].max()

    forecast_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=FORECAST_HORIZON,
        freq="D"
    )

    for date, forecast_value in zip(
        forecast_dates,
        prediction
    ):
        final_forecasts.append({
            "Bar Name": bar,
            "Brand Name": brand,
            "Date": date,
            "Selected Model": model_name,
            "Forecast Consumption (ml)": max(
                0,
                forecast_value
            )
        })

final_forecasts = pd.DataFrame(final_forecasts)

display(final_forecasts.head(30))

print(final_forecasts.shape)

print(final_forecasts["Selected Model"].value_counts())



### 5.2 — Create a 30-day forecast summary


In [ ]:
forecast_summary = (
    final_forecasts
    .groupby(
        ["Bar Name", "Brand Name", "Selected Model"],
        as_index=False
    )
    .agg(
        forecast_30d_ml=(
            "Forecast Consumption (ml)",
            "sum"
        ),
        forecast_daily_avg_ml=(
            "Forecast Consumption (ml)",
            "mean"
        ),
        forecast_daily_max_ml=(
            "Forecast Consumption (ml)",
            "max"
        )
    )
)

display(
    forecast_summary.head(20)
)


### 5.3 — Save the forecasting outputs


In [ ]:
if "PROJECT_ROOT" not in globals():
    from pathlib import Path

    def _find_project_root(start: Path) -> Path:
        for p in [start] + list(start.parents):
            if (p / "data" / "raw" / "bar_inventory_data.csv").exists():
                return p
        return start

    PROJECT_ROOT = _find_project_root(Path.cwd())

import os

os.makedirs(
    str(PROJECT_ROOT / "data" / "processed"),
    exist_ok=True
)

rolling_results.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "rolling_backtest_results.csv"),
    index=False
)

rolling_summary.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "rolling_model_summary.csv")
)

final_model_selection.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "best_forecasting_models.csv"),
    index=False
)

final_forecasts.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "model_predictions.csv"),
    index=False
)

forecast_summary.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "forecast_summary.csv"),
    index=False
)

print("Forecasting outputs saved successfully.")



### 5.4 — Don't jump straight to safety stock

We need forecast error first. Example: forecast 300 ml/day, actual 450 ml/day -> error +150 ml. If that persists, using the raw point forecast as expected demand understates inventory. Safety stock must come from *historical forecast errors and their variability*, not from raw demand std.


### 5.5 — Generate historical forecast errors


In [ ]:
error_results = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    group = (
        group
        .sort_values("Date")
        .reset_index(drop=True)
    )

    splits = rolling_time_splits(
        group,
        forecast_horizon=30,
        n_splits=4,
        min_train_size=90
    )

    selected = final_model_selection[
        (final_model_selection["Bar Name"] == bar)
        & (final_model_selection["Brand Name"] == brand)
    ]

    if selected.empty:
        continue

    model_name = selected.iloc[0]["Model"]

    for fold, (train, test) in enumerate(
        splits,
        start=1
    ):

        prediction = generate_forecast(
            model_name,
            train,
            len(test)
        )

        for date, actual, forecast in zip(
            test["Date"],
            test["daily_consumption_ml"],
            prediction
        ):

            error_results.append({
                "Bar Name": bar,
                "Brand Name": brand,
                "Fold": fold,
                "Date": date,
                "Model": model_name,
                "Actual": actual,
                "Forecast": forecast,
                "Error": actual - forecast,
                "Absolute Error": abs(
                    actual - forecast
                )
            })

forecast_errors = pd.DataFrame(
    error_results
)

display(
    forecast_errors.head(20)
)


In [ ]:
error_summary = (
    forecast_errors
    .groupby(
        ["Bar Name", "Brand Name"],
        as_index=False
    )
    .agg(
        error_mean=("Error", "mean"),
        error_std=("Error", "std"),
        mae=("Absolute Error", "mean")
    )
)

display(error_summary.head(20))


### 8.1 Phase 5.2 — Forecast Uncertainty, Safety Stock, ROP, Par Level & Order Quantity

Turning the selected model into inventory decisions. Core formulas:

$$ SS = z \cdot \sigma_{daily} \cdot \sqrt{LT} $$
$$ ROP = Lead\text{-}Time\ Demand + SS $$
$$ Par = Demand_{LT + cycle} + SS $$
$$ Order\ Quantity = \max\left(0,\ Par - Inventory\ Position\right) $$

where $z$ comes from the target service level, $\sigma_{daily}$ is the selected model's daily forecast error (from the rolling backtest), and LT is the supplier lead time.


In [ ]:
from scipy.stats import norm

LEAD_TIME_DAYS = 7
ORDER_CYCLE_DAYS = 7
SERVICE_LEVEL = 0.95
FORECAST_HORIZON = 30
Z_SERVICE = norm.ppf(SERVICE_LEVEL)

print(f"Lead time: {LEAD_TIME_DAYS} days")
print(f"Review / order cycle: {ORDER_CYCLE_DAYS} days")
print(f"Service level: {SERVICE_LEVEL:.0%} (z = {Z_SERVICE:.3f})")
print(f"Forecast horizon: {FORECAST_HORIZON} days")


In [ ]:
rolling_errors = (
    rolling_results
    .dropna(subset=["WAPE"])
    .copy()
)

forecast_uncertainty = (
    rolling_errors
    .groupby(["Bar Name", "Brand Name", "Model"])
    .agg(
        sigma_daily=("RMSE", "mean"),
        n_windows=("RMSE", "count"),
        mean_WAPE=("WAPE", "mean")
    )
    .reset_index()
)

display(forecast_uncertainty.head())


In [ ]:
final_forecast_rows = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    sel = final_model_selection[
        (final_model_selection["Bar Name"] == bar) &
        (final_model_selection["Brand Name"] == brand)
    ].iloc[0]

    model = sel["Model"]

    group_sorted = (
        group
        .sort_values("Date")
        .reset_index(drop=True)
    )

    prediction = generate_forecast(
        model,
        group_sorted,
        FORECAST_HORIZON
    )

    fc_mean = float(np.mean(prediction))
    fc_total = float(np.sum(prediction))

    unc = forecast_uncertainty[
        (forecast_uncertainty["Bar Name"] == bar) &
        (forecast_uncertainty["Brand Name"] == brand) &
        (forecast_uncertainty["Model"] == model)
    ]

    if len(unc) > 0:
        sigma_daily = float(unc["sigma_daily"].iloc[0])
        n_windows = int(unc["n_windows"].iloc[0])
    else:
        sigma_daily = float(group_sorted["daily_consumption_ml"].std())
        n_windows = 0

    final_forecast_rows.append({
        "Bar Name": bar,
        "Brand Name": brand,
        "Model": model,
        "mean_daily_forecast_ml": fc_mean,
        "forecast_total_30d_ml": fc_total,
        "sigma_daily_ml": sigma_daily,
        "n_validation_windows": n_windows,
    })

final_forecast = pd.DataFrame(final_forecast_rows)

display(final_forecast.head())


In [ ]:
final_forecast["lead_time_demand_ml"] = (
    final_forecast["mean_daily_forecast_ml"]
    * LEAD_TIME_DAYS
)

final_forecast["sigma_lead_time_ml"] = (
    final_forecast["sigma_daily_ml"]
    * np.sqrt(LEAD_TIME_DAYS)
)

final_forecast["safety_stock_ml"] = (
    Z_SERVICE
    * final_forecast["sigma_lead_time_ml"]
)

final_forecast["reorder_point_ml"] = (
    final_forecast["lead_time_demand_ml"]
    + final_forecast["safety_stock_ml"]
)

final_forecast["par_level_ml"] = (
    final_forecast["mean_daily_forecast_ml"]
    * (LEAD_TIME_DAYS + ORDER_CYCLE_DAYS)
    + final_forecast["safety_stock_ml"]
)

display(
    final_forecast[
        [
            "Bar Name",
            "Brand Name",
            "Model",
            "lead_time_demand_ml",
            "safety_stock_ml",
            "reorder_point_ml",
            "par_level_ml"
        ]
    ].head(20)
)


In [ ]:
last_closing = (
    daily
    .sort_values("Date")
    .groupby(["Bar Name", "Brand Name"])
    ["closing_inventory_ml"]
    .last()
)

final_forecast["current_inventory_ml"] = (
    final_forecast.apply(
        lambda row: float(last_closing.get(
            (row["Bar Name"], row["Brand Name"]),
            0.0
        )),
        axis=1
    )
    .fillna(0.0)
)

final_forecast["recommended_order_ml"] = (
    final_forecast["par_level_ml"]
    - final_forecast["current_inventory_ml"]
).clip(lower=0)

display(
    final_forecast.sort_values(
        "recommended_order_ml",
        ascending=False
    )[
        [
            "Bar Name",
            "Brand Name",
            "Model",
            "par_level_ml",
            "current_inventory_ml",
            "recommended_order_ml"
        ]
    ].head(20)
)


In [ ]:
rng = np.random.default_rng(42)
N_SIMS = 500

simulation_rows = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    row = final_forecast[
        (final_forecast["Bar Name"] == bar) &
        (final_forecast["Brand Name"] == brand)
    ].iloc[0]

    par = float(row["par_level_ml"])
    inv0 = float(row["current_inventory_ml"])
    order = float(row["recommended_order_ml"])

    series_demand = group["daily_consumption_ml"].values
    p_active = float((series_demand > 0).mean())
    active_vals = series_demand[series_demand > 0]

    demand_matrix = np.zeros((N_SIMS, FORECAST_HORIZON))
    mask = rng.random((N_SIMS, FORECAST_HORIZON)) < p_active
    amt = rng.integers(
        0,
        len(active_vals),
        size=(N_SIMS, FORECAST_HORIZON)
    )
    demand_matrix[mask] = active_vals[amt[mask]]

    inv = np.full(N_SIMS, inv0 + order)
    stockouts = np.zeros(N_SIMS)

    for d in range(FORECAST_HORIZON):
        if d % ORDER_CYCLE_DAYS == 0:
            inv = np.full(N_SIMS, par)

        inv = inv - demand_matrix[:, d]
        stockouts[inv < 0] += 1
        inv = np.maximum(inv, 0)

    simulation_rows.append({
        "Bar Name": bar,
        "Brand Name": brand,
        "mean_stockout_days": float(stockouts.mean()),
        "achieved_service_level": float(
            1 - stockouts.mean() / FORECAST_HORIZON,
        ),
        "share_no_stockout": float((stockouts == 0).mean()),
        "mean_ending_inventory_ml": float(inv.mean()),
    })

inventory_simulation = pd.DataFrame(simulation_rows)

display(inventory_simulation.head())

print("Target service level:", f"{SERVICE_LEVEL:.0%}")
print("Mean achieved service:", f"{inventory_simulation['achieved_service_level'].mean():.1%}")


In [ ]:
inventory_planning = final_forecast.merge(
    inventory_simulation,
    on=["Bar Name", "Brand Name"],
    how="left"
)

inventory_planning.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "inventory_planning.csv"),
    index=False
)

inventory_simulation.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "inventory_simulation.csv"),
    index=False
)

print("Saved inventory_planning.csv and inventory_simulation.csv")

display(
    inventory_planning.sort_values(
        "recommended_order_ml",
        ascending=False
    )[
        [
            "Bar Name",
            "Brand Name",
            "Model",
            "safety_stock_ml",
            "reorder_point_ml",
            "par_level_ml",
            "current_inventory_ml",
            "recommended_order_ml",
            "achieved_service_level"
        ]
    ].head(15)
)


## 10. Phase 6 - Stockout Analysis, Forecast-Error Uncertainty & Inventory Policy

Three fixes over the previous first-pass policy:

1. **Closing = 0 is only a stockout *candidate*, not proof** - we inspect the raw transaction history before claiming demand is uncensored (Phase 6.1).
2. **Uncertainty comes from actual forecast errors** (mean_error / sigma_error / MAE), not mean RMSE, which mixes bias and noise (Phase 6.2).
3. **The simulation must honour lead time and inventory position** (on-hand + on-order), not reset inventory to par every review (Phase 6.5).


### 10.1 Historical Stockout / Censoring Analysis

A record where Closing = 0 while Consumed > 0 is a candidate for censored demand (stock ran out and may have limited what could be sold). It is not proof of censoring, but it tells us how often the demand series may be truncated.


In [ ]:
stockout_candidates = df.copy()

stockout_candidates["Closing Near Zero"] = (
    stockout_candidates["Closing Balance (ml)"] <= 0.01
)

stockout_candidates["Consumed Positive"] = (
    stockout_candidates["Consumed (ml)"] > 0
)

stockout_candidates["Potential Stockout"] = (
    stockout_candidates["Closing Near Zero"]
    & stockout_candidates["Consumed Positive"]
)

print(
    "Potential stockout records:",
    stockout_candidates["Potential Stockout"].sum()
)

print(
    "Total records:",
    len(stockout_candidates)
)


In [ ]:
stockout_summary = (
    stockout_candidates[
        stockout_candidates["Potential Stockout"]
    ]
    .groupby(
        ["Bar Name", "Brand Name"],
        as_index=False
    )
    .agg(
        potential_stockout_records=(
            "Potential Stockout",
            "sum"
        ),
        total_consumption_ml=(
            "Consumed (ml)",
            "sum"
        )
    )
    .sort_values(
        "potential_stockout_records",
        ascending=False
    )
)

display(stockout_summary.head(20))


In [ ]:
zero_inventory = df[
    df["Closing Balance (ml)"] <= 0.01
].copy()

print(
    "Records reaching zero inventory:",
    len(zero_inventory)
)

display(
    zero_inventory[
        [
            "Date Time Served",
            "Bar Name",
            "Brand Name",
            "Opening Balance (ml)",
            "Purchase (ml)",
            "Consumed (ml)",
            "Closing Balance (ml)"
        ]
    ].head(30)
)


### 10.2 Forecast-Error Uncertainty

RMSE mixes systematic bias and random variation. Instead, use the selected model actual errors directly: mean_error (bias), sigma_error (variability), MAE (typical absolute error).


In [ ]:
selected_error_summary = (
    forecast_errors
    .groupby(
        ["Bar Name", "Brand Name", "Model"],
        as_index=False
    )
    .agg(
        mean_error=("Error", "mean"),
        sigma_error=("Error", "std"),
        mae=("Absolute Error", "mean"),
        observations=("Error", "count")
    )
)

selected_uncertainty = (
    final_model_selection[
        [
            "Bar Name",
            "Brand Name",
            "Model"
        ]
    ]
    .merge(
        selected_error_summary,
        on=[
            "Bar Name",
            "Brand Name",
            "Model"
        ],
        how="left"
    )
)

display(
    selected_uncertainty.head(20)
)


### 10.3 Safety Stock, ROP & Par Level

$$ SS = z * sigma_error * sqrt(LT) $$
$$ ROP = LT demand + SS $$
$$ Par = demand(LT + cycle) + SS $$


In [ ]:
final_forecast = final_forecast.merge(
    selected_uncertainty[
        [
            "Bar Name",
            "Brand Name",
            "sigma_error",
            "mean_error",
            "mae"
        ]
    ],
    on=["Bar Name", "Brand Name"],
    how="left"
)

final_forecast["sigma_lead_time_ml"] = (
    final_forecast["sigma_error"]
    * np.sqrt(LEAD_TIME_DAYS)
)

final_forecast["safety_stock_ml"] = (
    Z_SERVICE
    * final_forecast["sigma_lead_time_ml"]
)

final_forecast["lead_time_demand_ml"] = (
    final_forecast["mean_daily_forecast_ml"]
    * LEAD_TIME_DAYS
)

final_forecast["reorder_point_ml"] = (
    final_forecast["lead_time_demand_ml"]
    + final_forecast["safety_stock_ml"]
)

final_forecast["par_level_ml"] = (
    final_forecast["mean_daily_forecast_ml"]
    * (LEAD_TIME_DAYS + ORDER_CYCLE_DAYS)
    + final_forecast["safety_stock_ml"]
)

display(
    final_forecast[
        [
            "Bar Name",
            "Brand Name",
            "Model",
            "mean_error",
            "sigma_error",
            "safety_stock_ml",
            "reorder_point_ml",
            "par_level_ml"
        ]
    ].head(20)
)


### 10.4 Inventory Policy Simulation

A periodic-review policy with real lead time. Every ORder_CYCLE_DAYS days we compare **inventory position = on-hand + on-order** against Par and place an order if short; the order arrives LEAD_TIME_DAYS later. Demand is bootstrapped from historical active-day quantity frequency.


In [ ]:
rng = np.random.default_rng(42)

N_SIMS = 500

simulation_rows = []

for (bar, brand), group in daily.groupby(
    ["Bar Name", "Brand Name"]
):

    policy = final_forecast[
        (final_forecast["Bar Name"] == bar)
        & (final_forecast["Brand Name"] == brand)
    ].iloc[0]

    par = float(policy["par_level_ml"])
    initial_inventory = float(
        policy["current_inventory_ml"]
    )

    series_demand = (
        group["daily_consumption_ml"]
        .values
    )

    p_active = float(
        (series_demand > 0).mean()
    )

    active_values = (
        series_demand[
            series_demand > 0
        ]
    )

    if len(active_values) == 0:
        continue

    demand_matrix = np.zeros(
        (N_SIMS, FORECAST_HORIZON)
    )

    mask = (
        rng.random(
            (N_SIMS, FORECAST_HORIZON)
        ) < p_active
    )

    sampled_indices = rng.integers(
        0,
        len(active_values),
        size=(
            N_SIMS,
            FORECAST_HORIZON
        )
    )

    demand_matrix[mask] = (
        active_values[sampled_indices[mask]]
    )

    inventory = np.full(
        N_SIMS,
        initial_inventory
    )

    pipeline = np.zeros(
        (
            N_SIMS,
            FORECAST_HORIZON + LEAD_TIME_DAYS + 1
        )
    )

    stockout_days = np.zeros(
        N_SIMS
    )

    lost_demand = np.zeros(
        N_SIMS
    )

    order_count = np.zeros(
        N_SIMS
    )

    total_order_qty = np.zeros(
        N_SIMS
    )

    for day in range(FORECAST_HORIZON):

        inventory += pipeline[:, day]

        demand = demand_matrix[:, day]

        fulfilled = np.minimum(
            inventory,
            demand
        )

        unmet = np.maximum(
            demand - inventory,
            0
        )

        inventory -= fulfilled

        lost_demand += unmet

        stockout_days += (
            inventory <= 0
        ).astype(float)

        if day % ORDER_CYCLE_DAYS == 0:

            on_order = (
                pipeline[:, day + 1:]
                .sum(axis=1)
            )

            inventory_position = (
                inventory + on_order
            )

            order_qty = np.maximum(
                par - inventory_position,
                0
            )

            arrival_day = (
                day + LEAD_TIME_DAYS
            )

            if arrival_day < pipeline.shape[1]:

                pipeline[:, arrival_day] += (
                    order_qty
                )

            order_count += (
                order_qty > 0
            ).astype(float)

            total_order_qty += order_qty

    simulation_rows.append({

        "Bar Name": bar,
        "Brand Name": brand,

        "mean_stockout_days":
            float(stockout_days.mean()),

        "service_level":
            float(
                1
                - (
                    lost_demand.sum()
                    /
                    demand_matrix.sum()
                    if demand_matrix.sum() > 0
                    else 0
                )
            ),

        "share_no_stockout":
            float(
                (stockout_days == 0).mean()
            ),

        "mean_lost_demand_ml":
            float(
                lost_demand.mean()
            ),

        "mean_ending_inventory_ml":
            float(
                inventory.mean()
            ),

        "mean_orders":
            float(
                order_count.mean()
            ),

        "mean_order_quantity_ml":
            float(
                total_order_qty.mean()
                /
                np.maximum(
                    order_count.mean(),
                    1
                )
            )
    })

inventory_simulation = pd.DataFrame(
    simulation_rows
)

display(
    inventory_simulation.head(20)
)


### 10.5 Policy Comparison & Saved Outputs

Compare simulated service against the target, and quantify over/under-stocking across the 96 Bar x Brand series.


In [ ]:
inventory_planning = final_forecast.merge(
    inventory_simulation,
    on=["Bar Name", "Brand Name"],
    how="left"
)

inventory_planning.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "inventory_planning.csv"),
    index=False
)

inventory_simulation.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "inventory_simulation.csv"),
    index=False
)

print("inventory_planning.csv and inventory_simulation.csv saved")

print("Target service level:", f"{SERVICE_LEVEL:.0%}")
print("Mean achieved service:", f'{inventory_simulation["service_level"].mean():.1%}')

series_below_target = (
    inventory_simulation["service_level"] < SERVICE_LEVEL
).sum()
print("Series below target service:", series_below_target, "of", len(inventory_simulation))
print("Mean stockout days:", round(inventory_simulation["mean_stockout_days"].mean(), 2))
print("Mean lost demand (ml):", round(inventory_simulation["mean_lost_demand_ml"].mean(), 1))
print("Mean orders placed:", round(inventory_simulation["mean_orders"].mean(), 2))
print("Mean order quantity (ml):", round(inventory_simulation["mean_order_quantity_ml"].mean(), 0))
print("Mean ending inventory (ml):", round(inventory_simulation["mean_ending_inventory_ml"].mean(), 0))

display(
    inventory_simulation.sort_values(
        "service_level"
    ).head(10)
)


## 11. Phase 7 - Zero-Forecast Correction, Policy Re-run & Sensitivity

Problem identified in Phase 6: some series forecast 0 ml/30d (selected model = MA on a trailing stretch of zero days) despite demonstrable historical consumption. Zero forecast -> zero lead-time demand -> ROP = SS -> tiny par -> stockout in simulation.

Fix = a **forecast sanity floor**, applied transparently with an adjustment flag, then a Policy A vs Policy B simulation comparison, then a lead-time/service sensitivity matrix.


### 11.1 Identify suspicious zero forecasts

Which series have forecast_daily_ml <= 0, and what does their history actually show?


In [ ]:
planning = final_forecast[[
    "Bar Name", "Brand Name", "Model",
    "mean_daily_forecast_ml", "mean_error", "sigma_error",
    "safety_stock_ml", "reorder_point_ml", "par_level_ml",
    "current_inventory_ml"
]].rename(columns={
    "mean_daily_forecast_ml": "forecast_daily_ml",
    "reorder_point_ml": "rop_ml",
})

zero_forecast_diagnostics = planning[
    planning["forecast_daily_ml"] <= 0
].copy()

print("Zero-forecast series:", len(zero_forecast_diagnostics))

display(
    zero_forecast_diagnostics[
        [
            "Bar Name",
            "Brand Name",
            "Model",
            "forecast_daily_ml",
            "sigma_error",
            "safety_stock_ml",
            "rop_ml",
            "par_level_ml"
        ]
    ].sort_values("Brand Name")
)


In [ ]:
zero_keys = zero_forecast_diagnostics[[
    "Bar Name", "Brand Name"
]]

zero_history = daily.merge(
    zero_keys,
    on=["Bar Name", "Brand Name"],
    how="inner"
)

zero_history_summary = (
    zero_history
    .groupby(["Bar Name", "Brand Name"])
    .agg(
        total_consumption_ml=("daily_consumption_ml", "sum"),
        active_days=("daily_consumption_ml", lambda x: (x > 0).sum()),
        total_days=("daily_consumption_ml", "count"),
        mean_daily_ml=("daily_consumption_ml", "mean"),
        active_mean_ml=(
            "daily_consumption_ml",
            lambda x: x[x > 0].mean() if (x > 0).any() else 0
        ),
        recent_30d_mean_ml=(
            "daily_consumption_ml",
            lambda x: x.tail(30).mean()
        ),
    )
    .reset_index()
)

zero_history_summary["active_day_pct"] = (
    zero_history_summary["active_days"]
    / zero_history_summary["total_days"]
    * 100
)

zero_forecast_diagnostics = zero_forecast_diagnostics.merge(
    zero_history_summary,
    on=["Bar Name", "Brand Name"],
    how="left"
)

display(
    zero_history_summary.sort_values(
        "total_consumption_ml",
        ascending=False
    )
)


### 11.2 Forecast sanity floor

The rule: a model may not emit a zero operational forecast when the Bar x Brand has shown recurring recent demand. The floor is **conservative** - min(30-day recent mean, intermittent mean) - not the largest historical value.


In [ ]:
def apply_forecast_sanity_floor(
    forecast,
    demand,
    minimum_active_days=3,
    recent_window=30
):
    forecast = float(forecast)

    if forecast > 0:
        return forecast

    active_demand = demand[demand > 0]

    if len(active_demand) < minimum_active_days:
        return 0.0

    recent = demand.tail(recent_window)
    recent_mean = recent.mean()

    active_probability = (demand > 0).mean()
    active_mean = active_demand.mean()

    intermittent_mean = active_probability * active_mean

    candidates = [
        recent_mean,
        intermittent_mean,
    ]

    positive_candidates = [
        value for value in candidates
        if value > 0
    ]

    if not positive_candidates:
        return 0.0

    return min(positive_candidates)

series_demand = {
    (bar, brand): group["daily_consumption_ml"]
    for (bar, brand), group in daily.groupby(
        ["Bar Name", "Brand Name"]
    )
}

planning["forecast_original_ml"] = planning["forecast_daily_ml"]

planning["forecast_adjusted_ml"] = planning.apply(
    lambda row: apply_forecast_sanity_floor(
        row["forecast_daily_ml"],
        series_demand[(row["Bar Name"], row["Brand Name"])]
    ),
    axis=1
)

planning["forecast_adjustment_flag"] = (
    planning["forecast_adjusted_ml"]
    != planning["forecast_original_ml"]
)

print(
    "Series with adjusted (floored) forecast:",
    int(planning["forecast_adjustment_flag"].sum()),
    "of",
    len(planning)
)

display(
    planning[
        planning["forecast_adjustment_flag"]
    ][
        [
            "Bar Name",
            "Brand Name",
            "Model",
            "forecast_original_ml",
            "forecast_adjusted_ml",
        ]
    ]
)


### 11.3 Recalculate the inventory policy on the adjusted forecast

Adjustment flag is explicit so the report can disclose the intervention. Original par kept as par_original_ml for the Policy A comparison.


In [ ]:
# Policy A (original) stored before overwriting
planning["lead_time_demand_original_ml"] = (
    planning["forecast_daily_ml"] * LEAD_TIME_DAYS
)
planning["rop_original_ml"] = (
    planning["lead_time_demand_original_ml"]
    + planning["safety_stock_ml"]
)
planning["par_original_ml"] = (
    planning["forecast_daily_ml"]
    * (LEAD_TIME_DAYS + ORDER_CYCLE_DAYS)
    + planning["safety_stock_ml"]
)

# Policy B (floored forecast)
planning["lead_time_demand_ml"] = (
    planning["forecast_adjusted_ml"] * LEAD_TIME_DAYS
)
planning["rop_ml"] = (
    planning["lead_time_demand_ml"]
    + planning["safety_stock_ml"]
)
planning["par_level_ml"] = (
    planning["forecast_adjusted_ml"]
    * (LEAD_TIME_DAYS + ORDER_CYCLE_DAYS)
    + planning["safety_stock_ml"]
)
planning["recommended_order_ml"] = (
    planning["par_level_ml"] - planning["current_inventory_ml"]
).clip(lower=0)

display(
    planning[
        planning["forecast_adjustment_flag"]
    ][
        [
            "Bar Name",
            "Brand Name",
            "forecast_original_ml",
            "forecast_adjusted_ml",
            "safety_stock_ml",
            "rop_ml",
            "par_level_ml",
            "recommended_order_ml"
        ]
    ]
)


### 11.4 Lead-time-aware simulation as a reusable function

Same engine as Phase 6.5, factorised so Policy A, Policy B and every sensitivity combination run through identical scenarios (same seed).


In [ ]:
SERIES_GROUPS = [
    (key, group)
    for key, group in daily.groupby(
        ["Bar Name", "Brand Name"]
    )
]

def run_inventory_simulation(
    policy,
    par_col="par_level_ml",
    groups=SERIES_GROUPS,
    seed=42,
    n_sims=500,
):
    rng = np.random.default_rng(seed)
    simulation_rows = []
    policy_index = policy.set_index(
        ["Bar Name", "Brand Name"]
    )

    for (bar, brand), group in groups:
        if (bar, brand) not in policy_index.index:
            continue

        policy_row = policy_index.loc[(bar, brand)]

        par = float(policy_row[par_col])
        initial_inventory = float(
            policy_row["current_inventory_ml"]
        )

        series_demand = group["daily_consumption_ml"].values
        p_active = float((series_demand > 0).mean())
        active_values = series_demand[series_demand > 0]

        if len(active_values) == 0:
            continue

        demand_matrix = np.zeros(
            (n_sims, FORECAST_HORIZON)
        )
        mask = (
            rng.random((n_sims, FORECAST_HORIZON)) < p_active
        )
        sampled_indices = rng.integers(
            0, len(active_values),
            size=(n_sims, FORECAST_HORIZON)
        )
        demand_matrix[mask] = (
            active_values[sampled_indices[mask]]
        )

        inventory = np.full(n_sims, initial_inventory)
        pipeline = np.zeros(
            (n_sims, FORECAST_HORIZON + LEAD_TIME_DAYS + 1)
        )
        stockout_days = np.zeros(n_sims)
        lost_demand = np.zeros(n_sims)
        fulfilled_total = np.zeros(n_sims)
        on_hand_sum = np.zeros(n_sims)
        order_count = np.zeros(n_sims)
        total_order_qty = np.zeros(n_sims)

        for day in range(FORECAST_HORIZON):
            inventory += pipeline[:, day]

            demand = demand_matrix[:, day]
            fulfilled = np.minimum(inventory, demand)
            unmet = np.maximum(demand - inventory, 0)

            inventory -= fulfilled
            fulfilled_total += fulfilled
            on_hand_sum += inventory
            lost_demand += unmet
            stockout_days += (
                inventory <= 0
            ).astype(float)

            if day % ORDER_CYCLE_DAYS == 0:
                on_order = pipeline[:, day + 1:].sum(axis=1)
                inventory_position = inventory + on_order
                order_qty = np.maximum(
                    par - inventory_position, 0
                )
                arrival_day = day + LEAD_TIME_DAYS
                if arrival_day < pipeline.shape[1]:
                    pipeline[:, arrival_day] += order_qty
                order_count += (order_qty > 0).astype(float)
                total_order_qty += order_qty

        total_demand = demand_matrix.sum()
        service = 1 - (
            lost_demand.sum() / total_demand
            if total_demand > 0 else 0
        )
        avg_on_hand = (
            on_hand_sum / FORECAST_HORIZON
        ).mean()

        simulation_rows.append({
            "Bar Name": bar,
            "Brand Name": brand,
            "service_level": float(service),
            "mean_stockout_days": float(
                stockout_days.mean()
            ),
            "mean_lost_demand_ml": float(
                lost_demand.mean()
            ),
            "mean_ending_inventory_ml": float(
                inventory.mean()
            ),
            "mean_on_hand_ml": float(avg_on_hand),
            "share_no_stockout": float(
                (stockout_days == 0).mean()
            ),
            "mean_orders": float(order_count.mean()),
            "mean_order_quantity_ml": float(
                total_order_qty.mean()
                / np.maximum(order_count.mean(), 1)
            ),
            "mean_fulfilled_ml": float(
                fulfilled_total.mean()
            ),
        })

    sim = pd.DataFrame(simulation_rows)
    sim["mean_turnover"] = (
        sim["mean_fulfilled_ml"]
        / sim["mean_on_hand_ml"].clip(lower=1e-6)
    )
    return sim


### 11.5 Policy A (original) vs Policy B (sanity floor)

Same demand scenarios, same service target - only the par/ROP inputs differ.


In [ ]:
sim_original = run_inventory_simulation(
    planning,
    par_col="par_original_ml",
)
sim_adjusted = run_inventory_simulation(
    planning,
    par_col="par_level_ml",
)

metrics = [
    ("Mean service level", "service_level", "mean"),
    ("Median service level", "service_level", "median"),
    ("Series below 95% target", "service_level", "below"),
    ("Mean stockout days", "mean_stockout_days", "mean"),
    ("Mean lost demand (ml)", "mean_lost_demand_ml", "mean"),
    ("Mean ending inventory (ml)", "mean_ending_inventory_ml", "mean"),
    ("Mean on-hand inventory (ml)", "mean_on_hand_ml", "mean"),
    ("Mean orders", "mean_orders", "mean"),
    ("Mean order qty (ml)", "mean_order_quantity_ml", "mean"),
    ("Mean turnover", "mean_turnover", "mean"),
]

comparison_rows = []
for label, col, how in metrics:
    if how == "mean":
        a = sim_original[col].mean()
        b = sim_adjusted[col].mean()
    elif how == "median":
        a = sim_original[col].median()
        b = sim_adjusted[col].median()
    else:
        a = (sim_original[col] < 0.95).sum()
        b = (sim_adjusted[col] < 0.95).sum()
    comparison_rows.append({
        "Metric": label,
        "Policy A (original)": round(a, 3),
        "Policy B (floored)": round(b, 3),
    })

policy_comparison = pd.DataFrame(comparison_rows)
display(policy_comparison)

inventory_simulation = sim_adjusted
planning = planning.merge(
    sim_adjusted[[
        "Bar Name", "Brand Name", "service_level",
        "mean_stockout_days", "mean_lost_demand_ml",
        "mean_ending_inventory_ml", "mean_order_quantity_ml",
    ]],
    on=["Bar Name", "Brand Name"],
    how="left"
)


### 11.6 Save Phase 7 outputs


In [ ]:
out = PROJECT_ROOT / "data" / "processed"

planning.to_csv(
    str(out / "inventory_planning.csv"),
    index=False
)
sim_adjusted.to_csv(
    str(out / "inventory_simulation.csv"),
    index=False
)
zero_forecast_diagnostics.to_csv(
    str(out / "zero_forecast_diagnostics.csv"),
    index=False
)
policy_comparison.to_csv(
    str(out / "policy_comparison.csv"),
    index=False
)
print("Saved: inventory_planning.csv, inventory_simulation.csv, zero_forecast_diagnostics.csv, policy_comparison.csv")


### 11.7 Policy sensitivity matrix

Lead time x target service, all on the corrected (floored) policy. Same simulation engine.


In [ ]:
from scipy.stats import norm

sens_rows = []

for lt in [2, 3, 5, 7]:
    for sl in [0.90, 0.95, 0.99]:
        tmp = planning.copy()
        z = norm.ppf(sl)

        tmp["safety_stock_ml"] = (
            z * tmp["sigma_error"] * np.sqrt(lt)
        )
        tmp["lead_time_demand_ml"] = (
            tmp["forecast_adjusted_ml"] * lt
        )
        tmp["rop_ml"] = (
            tmp["lead_time_demand_ml"]
            + tmp["safety_stock_ml"]
        )
        tmp["par_level_ml"] = (
            tmp["forecast_adjusted_ml"]
            * (lt + ORDER_CYCLE_DAYS)
            + tmp["safety_stock_ml"]
        )

        sim = run_inventory_simulation(
            tmp,
            par_col="par_level_ml",
            seed=42,
        )

        sens_rows.append({
            "lead_time_days": lt,
            "target_service": sl,
            "mean_service": round(
                sim["service_level"].mean(), 4
            ),
            "median_service": round(
                sim["service_level"].median(), 4
            ),
            "series_below_target": int(
                (sim["service_level"] < sl).sum()
            ),
            "mean_stockout_days": round(
                sim["mean_stockout_days"].mean(), 3
            ),
            "mean_lost_demand_ml": round(
                sim["mean_lost_demand_ml"].mean(), 1
            ),
            "mean_on_hand_ml": round(
                sim["mean_on_hand_ml"].mean(), 0
            ),
            "mean_ending_inventory_ml": round(
                sim["mean_ending_inventory_ml"].mean(), 0
            ),
            "mean_orders": round(
                sim["mean_orders"].mean(), 2
            ),
            "mean_order_qty_ml": round(
                sim["mean_order_quantity_ml"].mean(), 0
            ),
            "mean_turnover": round(
                sim["mean_turnover"].mean(), 2
            ),
        })

inventory_policy_sensitivity = pd.DataFrame(
    sens_rows
)

display(inventory_policy_sensitivity)

svc = inventory_policy_sensitivity.pivot(
    index="lead_time_days",
    columns="target_service",
    values="mean_service",
)
display(svc)

inventory_policy_sensitivity.to_csv(
    str(out / "inventory_policy_sensitivity.csv"),
    index=False
)
print("Saved: inventory_policy_sensitivity.csv")


## 12. Phase 8 - Final Validation, KPIs & Business Interpretation

Before writing the report we prove the pipeline is internally consistent: every output file exists, no impossible inventory values, forecasts agree with the model selection, and the simulation obeys physical conservation (opening + arrivals - fulfilled = closing; fulfilled + lost = demand).


### 12.1 Output-file inventory

In [ ]:
expected_files = [
    "daily_bar_consumption.csv",
    "series_summary.csv",
    "intermittent_demand_analysis.csv",
    "forecast_model_results.csv",
    "best_forecasting_models.csv",
    "rolling_backtest_results.csv",
    "rolling_model_summary.csv",
    "model_predictions.csv",
    "forecast_summary.csv",
    "inventory_planning.csv",
    "inventory_simulation.csv",
    "zero_forecast_diagnostics.csv",
    "policy_comparison.csv",
    "inventory_policy_sensitivity.csv",
]

data_dir = PROJECT_ROOT / "data" / "processed"
file_check = []
for f in expected_files:
    p = data_dir / f
    if p.exists():
        t = pd.read_csv(p)
        file_check.append({
            "file": f,
            "exists": True,
            "rows": t.shape[0],
            "cols": t.shape[1],
        })
    else:
        file_check.append({
            "file": f,
            "exists": False,
            "rows": None,
            "cols": None,
        })

output_inventory = pd.DataFrame(file_check)
display(output_inventory)
print("Missing files:", output_inventory["exists"].value_counts().get(False, 0))


### 12.2 Impossible / negative inventory-policy values

In [ ]:
policy_cols = [
    "forecast_original_ml", "forecast_adjusted_ml",
    "mean_daily_forecast_ml" if "mean_daily_forecast_ml" in planning else "forecast_daily_ml",
    "sigma_error", "safety_stock_ml",
    "lead_time_demand_ml", "rop_ml", "par_level_ml",
    "recommended_order_ml", "current_inventory_ml",
]

neg = []
nan = []
for c in policy_cols:
    if c not in planning.columns:
        continue
    n_neg = int((planning[c] < -1e-9).sum())
    n_nan = int(planning[c].isna().sum())
    if n_neg or n_nan:
        neg.append(c)
        nan.append(c)
print("Columns with negative values:", neg if neg else "NONE")
print("Columns with NaN values:", nan if nan else "NONE")

structural = pd.DataFrame({
    "check": [
        "safety_stock_ml >= 0",
        "lead_time_demand_ml >= 0",
        "rop_ml >= safety_stock_ml",
        "par_level_ml >= rop_ml",
        "recommended_order_ml >= 0",
        "current_inventory_ml >= 0",
        "par_level_ml finite",
    ],
    "violations": [
        int((planning["safety_stock_ml"] < -1e-9).sum()),
        int((planning["lead_time_demand_ml"] < -1e-9).sum()),
        int((planning["rop_ml"] + 1e-9 < planning["safety_stock_ml"]).sum()),
        int((planning["par_level_ml"] + 1e-9 < planning["rop_ml"]).sum()),
        int((planning["recommended_order_ml"] < -1e-9).sum()),
        int((planning["current_inventory_ml"] < -1e-9).sum()),
        int((~np.isfinite(planning["par_level_ml"])).sum()),
    ],
})
display(structural)


### 12.3 Forecast / model-selection consistency

In [ ]:
model_check = final_model_selection[
    ["Bar Name", "Brand Name", "Model"]
].merge(
    planning[["Bar Name", "Brand Name", "Model"]].rename(
        columns={"Model": "planning_model"}
    ),
    on=["Bar Name", "Brand Name"],
    how="outer"
)
model_check["model_mismatch"] = (
    model_check["Model"] != model_check["planning_model"]
)
print("Series with model-selection mismatch:",
      int(model_check["model_mismatch"].sum()), "of", len(model_check))

fc_check = forecast_summary[
    ["Bar Name", "Brand Name", "forecast_daily_avg_ml"]
].merge(
    planning[["Bar Name", "Brand Name", "forecast_original_ml"]],
    on=["Bar Name", "Brand Name"],
    how="outer"
)
fc_check["fc_diff"] = (
    fc_check["forecast_daily_avg_ml"] - fc_check["forecast_original_ml"]
).abs()
print("Max |forecast_summary vs planning| diff:",
      round(float(fc_check["fc_diff"].max()), 6))
print("Rows with diff > 1e-6:", int((fc_check["fc_diff"] > 1e-6).sum()))
print("final_forecasts rows:", final_forecasts.shape[0], "(expect 2880 = 96 x 30)")


### 12.4 Simulation KPI self-consistency

In [ ]:
sim = inventory_simulation.copy()
sim["demand_ml"] = sim["mean_fulfilled_ml"] + sim["mean_lost_demand_ml"]
sim["expected_service"] = (
    1 - sim["mean_lost_demand_ml"] / sim["demand_ml"].clip(lower=1e-9)
)
sim["service_gap"] = (
    sim["service_level"] - sim["expected_service"]
).abs()

print("Rows where service formula inconsistent (>1e-6):",
      int((sim["service_gap"] > 1e-6).sum()), "of", len(sim))
print("Max service_gap:", round(float(sim["service_gap"].max()), 9))
print("Negative lost demand:", int((sim["mean_lost_demand_ml"] < -1e-9).sum()))
print("Negative ending inventory:", int((sim["mean_ending_inventory_ml"] < -1e-9).sum()))
print("Negative on-hand:", int((sim["mean_on_hand_ml"] < -1e-9).sum()))
print("Orders outside [0, horizon/cycle]:", int(((sim["mean_orders"] < 0) | (sim["mean_orders"] > 5)).sum()))
display(sim[["Bar Name", "Brand Name", "service_level", "expected_service", "service_gap"]].head(10))


### 12.5 Daily-trace accounting validator

In [ ]:
def trace_series(key, par_col="par_level_ml", seed=7):
    bar, brand = key
    row = planning.set_index(
        ["Bar Name", "Brand Name"]
    ).loc[key]
    par = float(row[par_col])
    initial_inventory = float(row["current_inventory_ml"])

    group = dict(SERIES_GROUPS)[key]
    demand_series = group["daily_consumption_ml"].values
    p_active = float((demand_series > 0).mean())
    active_values = demand_series[demand_series > 0]
    if len(active_values) == 0:
        return None

    rng = np.random.default_rng(seed)
    demand = np.zeros(FORECAST_HORIZON)
    mask = rng.random(FORECAST_HORIZON) < p_active
    demand[mask] = rng.choice(active_values, size=int(mask.sum()))

    inventory = initial_inventory
    pipeline = np.zeros(FORECAST_HORIZON + LEAD_TIME_DAYS + 1)
    orders_placed = 0.0
    total_arrivals = 0.0
    stockout_days = 0
    lost = 0.0

    checks = {"fulfilled_plus_lost_equals_demand": 0, "negative_inventory_days": 0}
    for day in range(FORECAST_HORIZON):
        arrivals = pipeline[day]
        total_arrivals += arrivals
        inventory += arrivals

        d = demand[day]
        fulfilled = min(inventory, d)
        unmet = max(d - inventory, 0)
        inventory -= fulfilled
        lost += unmet
        stockout_days += int(inventory <= 0)

        if abs(fulfilled + unmet - d) > 1e-9:
            checks["fulfilled_plus_lost_equals_demand"] += 1
        if inventory < -1e-9:
            checks["negative_inventory_days"] += 1

        if day % ORDER_CYCLE_DAYS == 0:
            position = inventory + pipeline[day + 1:].sum()
            order_qty = max(par - position, 0)
            if order_qty > 0:
                orders_placed += order_qty
                arrival_day = day + LEAD_TIME_DAYS
                if arrival_day < len(pipeline):
                    pipeline[arrival_day] += order_qty

    open_orders = float(pipeline[FORECAST_HORIZON:].sum())
    return {
        **checks,
        "order_vs_arrival_balance": abs(orders_placed - total_arrivals - open_orders),
        "open_orders_end": round(open_orders, 2),
        "stockout_days": stockout_days,
        "lost_ml": round(lost, 2),
        "ending_inventory": round(inventory, 2),
    }

trace_targets = [
    ("Anderson's Bar", "Bacardi"),
    ("Anderson's Bar", "Budweiser"),
    ("Smith's Bar", "Yellow Tail"),
]
for t in trace_targets:
    r = trace_series(t)
    ok = all(v == 0 for k, v in r.items() if k not in ("stockout_days", "lost_ml", "ending_inventory", "order_vs_arrival_balance", "open_orders_end"))
    print(("PASS " if ok else "FAIL "), t, "->", r)


### 12.6 Final KPI tables

In [ ]:
kpi = planning.merge(
    inventory_simulation[[
        "Bar Name", "Brand Name",
        "mean_on_hand_ml", "mean_orders", "mean_turnover",
        "mean_fulfilled_ml",
    ]],
    on=["Bar Name", "Brand Name"],
    how="left"
)

high_risk = kpi.sort_values("service_level", ascending=True)[
    ["Bar Name", "Brand Name", "Model", "forecast_adjusted_ml",
     "safety_stock_ml", "rop_ml", "par_level_ml", "mean_on_hand_ml",
     "service_level", "mean_stockout_days", "mean_lost_demand_ml"]
].head(15)

inventory_heavy = kpi.sort_values("mean_on_hand_ml", ascending=False)[
    ["Bar Name", "Brand Name", "Model", "par_level_ml", "mean_on_hand_ml",
     "service_level", "mean_turnover"]
].head(15)

print("TOP 15 HIGHEST-RISK (lowest simulated service)")
display(high_risk)
print("TOP 15 INVENTORY-HEAVY (highest mean on-hand stock)")
display(inventory_heavy)

svc = inventory_simulation["service_level"]
dist = pd.DataFrame({
    "metric": ["p10", "p25", "median", "mean", "p75", "p90"],
    "service": [round(float(svc.quantile(q)), 4) for q in (0.10, 0.25, 0.50, 0.50, 0.75, 0.90)],
})
dist.loc[3, "service"] = round(float(svc.mean()), 4)
print("SERVICE DISTRIBUTION (96 Bar x Brand)")
display(dist)
print("Series below 90%:", int((svc < 0.90).sum()), "below 95%:", int((svc < 0.95).sum()), "below 99%:", int((svc < 0.99).sum()))


### 12.7 Management-level charts

In [ ]:
report_dir = PROJECT_ROOT / "report"
report_dir.mkdir(exist_ok=True)

svc_series = inventory_simulation["service_level"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.hist(svc_series, bins=20, color="#4C72B0", edgecolor="white")
ax.axvline(0.95, color="crimson", ls="--", lw=2, label="95% target")
ax.axvline(svc_series.mean(), color="black", ls=":", lw=2, label=f"mean = {svc_series.mean():.1%}")
ax.set_title("Simulated service level by Bar x Brand")
ax.set_xlabel("Service level"); ax.set_ylabel("Series count"); ax.legend()
fig.tight_layout(); fig.savefig(report_dir / "fig_service_distribution.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(9, 7))
top = high_risk.iloc[::-1]
ax.barh(
    top["Bar Name"] + " - " + top["Brand Name"],
    top["service_level"], color="#C44E52"
)
ax.axvline(0.95, color="black", ls="--", lw=1.5, label="95% target")
ax.set_xlim(0, 1)
ax.set_xlabel("Simulated service level")
ax.set_title("Highest-risk combinations (lowest service)")
ax.legend(); fig.tight_layout()
fig.savefig(report_dir / "fig_high_risk.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.scatter(
    kpi["mean_on_hand_ml"], kpi["service_level"],
    alpha=0.5, s=25, color="#55A868"
)
ax.axhline(0.95, color="crimson", ls="--", lw=1.5)
ax.set_xlabel("Mean on-hand inventory (ml)")
ax.set_ylabel("Simulated service level")
ax.set_title("Service vs inventory holding (trade-off)")
fig.tight_layout(); fig.savefig(report_dir / "fig_service_vs_inventory.png", dpi=120)
plt.show()


### 12.8 Save final outputs

In [ ]:
high_risk.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "high_risk_series.csv"),
    index=False
)
inventory_heavy.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "inventory_heavy_series.csv"),
    index=False
)
kpi.to_csv(
    str(PROJECT_ROOT / "data" / "processed" / "final_kpis.csv"),
    index=False
)
print("Saved: high_risk_series.csv, inventory_heavy_series.csv, final_kpis.csv")
